# NLP Assignment 3 – Part 1: Transformer NMT (French → English)

## 0. Imports

In [2]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'nmt_transformer'))

import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from config import *
from data import get_dataloaders
from model import TransformerNMT
from train import train, save_checkpoint, load_checkpoint
from inference import greedy_decode, beam_search
from utils import (
    compute_bleu, compute_bleu_dataset,
    plot_attention, plot_all_heads, plot_loss_curves, ids_to_tokens
)
from transformers import PreTrainedTokenizerFast
from datasets import load_from_disk

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
BEAM_SIZE = 4

def clean(text):
    """Strip BPE boundary markers from decoded text."""
    return ' '.join(text.replace('\u2581', ' ').split())

print(f'Device: {DEVICE}')

Device: cuda


## 1. Load Data

In [3]:
train_loader, val_loader, test_loader, src_tokenizer, tgt_tokenizer = get_dataloaders(
    DATA_PATH, TOKENIZER_FR_PATH, TOKENIZER_EN_PATH,
    batch_size=BATCH_SIZE, max_seq_len=MAX_SEQ_LEN,
    bos_id=BOS_ID, eos_id=EOS_ID, pad_id=PAD_ID,
)
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

ds = load_from_disk(DATA_PATH)
sample = ds['train'][0]
print(f"\nSample  FR: {sample['text_fr']}")
print(f"        EN: {sample['text_en']}")

Train batches : 272
Val   batches : 16
Test  batches : 16

Sample  FR: je suis dure .
        EN: i m tough .


## 2. Build Model

In [4]:
model = TransformerNMT(
    src_vocab_size=SRC_VOCAB_SIZE, tgt_vocab_size=TGT_VOCAB_SIZE,
    embed_dim=HIDDEN_SIZE,         max_seq_len=MAX_SEQ_LEN,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    num_heads=NUM_HEADS,           intermediate_dim=INTERMEDIATE_SIZE,
    dropout=DROPOUT,               pad_id=PAD_ID,
).to(DEVICE)

print('Weight-tied projection:', model.decoder.embedding.token_embedding.weight.shape)
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(model)

Weight-tied projection: torch.Size([3200, 32])
Total parameters: 294,784
TransformerNMT(
  (encoder): Encoder(
    (embedding): TransformerEmbedding(
      (token_embedding): Embedding(3200, 32)
      (position_embedding): Embedding(32, 32)
    )
    (layers): ModuleList(
      (0-2): 3 x EncoderLayer(
        (self_attn): MultiHeadAttention(
          (W_q): Linear(in_features=32, out_features=32, bias=False)
          (W_k): Linear(in_features=32, out_features=32, bias=False)
          (W_v): Linear(in_features=32, out_features=32, bias=False)
          (W_o): Linear(in_features=32, out_features=32, bias=False)
        )
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=32, out_features=128, bias=True)
            (1): ReLU()
            (2): Linear(in_features=128, out_features=32, bias=True)
            (3): Dropout(p=0.1, inplace=False)
          )
        )
        (add_norm1): addNorm(
          (norm): LayerNorm((32,), eps=1e-05, eleme

In [5]:
# Forward pass sanity check
src_ids, dec_input, targets = next(iter(train_loader))
src_ids, dec_input = src_ids.to(DEVICE), dec_input.to(DEVICE)
with torch.no_grad():
    logits, enc_attn, self_attn, cross_attn = model(src_ids, dec_input)
print(f'logits        : {logits.shape}')           # (B, tgt_len, 3200)
print(f'enc_attn[-1]  : {enc_attn[-1].shape}')     # (B, 4, src_len, src_len)
print(f'cross_attn[-1]: {cross_attn[-1].shape}')   # (B, 4, tgt_len, src_len)

logits        : torch.Size([32, 10, 3200])
enc_attn[-1]  : torch.Size([32, 4, 13, 13])
cross_attn[-1]: torch.Size([32, 4, 10, 13])


## 3. Train

In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses, val_losses = train(
    model, train_loader, val_loader, optimizer,
    pad_id=PAD_ID, max_epochs=MAX_EPOCHS,
    device=DEVICE, checkpoint_dir=CHECKPOINT_DIR,
)

Epoch  1/10  train_loss=6.3961  val_loss=3.6446
  Checkpoint saved → /ExtraStorage/NLP-Projects/assigment_3/nmt_transformer/../checkpoints/best_transformer.pt
Epoch  2/10  train_loss=3.4180  val_loss=3.0443
  Checkpoint saved → /ExtraStorage/NLP-Projects/assigment_3/nmt_transformer/../checkpoints/best_transformer.pt
Epoch  3/10  train_loss=3.0163  val_loss=2.8376
  Checkpoint saved → /ExtraStorage/NLP-Projects/assigment_3/nmt_transformer/../checkpoints/best_transformer.pt
Epoch  4/10  train_loss=2.8004  val_loss=2.7074
  Checkpoint saved → /ExtraStorage/NLP-Projects/assigment_3/nmt_transformer/../checkpoints/best_transformer.pt
Epoch  5/10  train_loss=2.6351  val_loss=2.5748
  Checkpoint saved → /ExtraStorage/NLP-Projects/assigment_3/nmt_transformer/../checkpoints/best_transformer.pt
Epoch  6/10  train_loss=2.5011  val_loss=2.4917
  Checkpoint saved → /ExtraStorage/NLP-Projects/assigment_3/nmt_transformer/../checkpoints/best_transformer.pt
Epoch  7/10  train_loss=2.3820  val_loss=2.434

In [7]:
fig = plot_loss_curves(train_losses, val_losses)
plt.savefig('loss_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print('Loss curve saved to loss_curves.png')

Loss curve saved to loss_curves.png


/tmp/ipykernel_21673/3083147556.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Save & Load Checkpoint

In [8]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
save_checkpoint(
    model, optimizer, MAX_EPOCHS, val_losses[-1],
    os.path.join(CHECKPOINT_DIR, 'final_transformer.pt'),
)

epoch, val_loss = load_checkpoint(
    model, optimizer,
    os.path.join(CHECKPOINT_DIR, 'best_transformer.pt'),
    DEVICE,
)
print(f'Best model: epoch {epoch}, val_loss={val_loss:.4f}')

  Checkpoint saved → /ExtraStorage/NLP-Projects/assigment_3/nmt_transformer/../checkpoints/final_transformer.pt
Loaded checkpoint from /ExtraStorage/NLP-Projects/assigment_3/nmt_transformer/../checkpoints/best_transformer.pt  (epoch 10, val_loss 2.2620)
Best model: epoch 10, val_loss=2.2620


## 5. Greedy vs. Beam Search

In [9]:
test_sentences = [
    'je suis dure .',
    'il est tr\u00e8s intelligent .',
    'bonjour , comment \u00e7a va ?',
    'elle aime lire des livres .',
    'nous allons \u00e0 l \' \u00e9cole demain .',
]

print(f"{'French':<40} {'Greedy':<30} Beam (k=4)")
print('-' * 100)
for fr in test_sentences:
    greedy_out, _ = greedy_decode(
        model, fr, src_tokenizer, tgt_tokenizer,
        MAX_SEQ_LEN, BOS_ID, EOS_ID, PAD_ID, DEVICE
    )
    beam_out, _, _ = beam_search(
        model, fr, src_tokenizer, tgt_tokenizer,
        BEAM_SIZE, MAX_SEQ_LEN, BOS_ID, EOS_ID, PAD_ID, DEVICE
    )
    print(f'{fr:<40} {clean(greedy_out):<30} {clean(beam_out)}')

French                                   Greedy                         Beam (k=4)
----------------------------------------------------------------------------------------------------
je suis dure .                           i m not .                      i m kind .
il est très intelligent .                he is a now .                  he is big .
bonjour , comment ça va ?                i m doing in the aren t ?      i m doing in the aren t ?
elle aime lire des livres .              she is just a her .            she is s .
nous allons à l ' école demain .         we re going to be of the out . we re going going .


## 6. BLEU Score

In [10]:
print('Computing BLEU on test set ...')
bleu_result, hypotheses, references = compute_bleu_dataset(
    model, ds['test'], src_tokenizer, tgt_tokenizer,
    beam_size=BEAM_SIZE, max_len=MAX_SEQ_LEN,
    bos_id=BOS_ID, eos_id=EOS_ID, pad_id=PAD_ID, device=DEVICE,
)
print(f'\nTest BLEU = {bleu_result.score:.2f}')
print(bleu_result)

print('\nSample predictions:')
for i in range(5):
    print(f'  REF : {references[i]}')
    print(f'  HYP : {clean(hypotheses[i])}')
    print()

Computing BLEU on test set ...

Test BLEU = 0.07
BLEU = 0.07 14.9/0.0/0.0/0.0 (BP = 1.000 ratio = 1.106 hyp_len = 3206 ref_len = 2898)

Sample predictions:
  REF : i m trustworthy .
  HYP : i m going to .

  REF : i m not miserable .
  HYP : i m not only .

  REF : i m going to take my car .
  HYP : i m going to going .

  REF : he s a cat lover .
  HYP : he s looking for me .

  REF : i m happy with that .
  HYP : i m go .



## 7. Attention Visualization

In [11]:
src_text = 'je suis dure .'
translation, pred_tokens, cross_attn_last = beam_search(
    model, src_text, src_tokenizer, tgt_tokenizer,
    BEAM_SIZE, MAX_SEQ_LEN, BOS_ID, EOS_ID, PAD_ID, DEVICE,
)
print(f'FR : {src_text}')
print(f'EN : {clean(translation)}')

src_ids_list = src_tokenizer.encode(src_text, add_special_tokens=False) + [EOS_ID]
src_labels   = ids_to_tokens(src_ids_list, src_tokenizer) + ['</s>']
tgt_labels   = ids_to_tokens(pred_tokens, tgt_tokenizer)

if cross_attn_last is not None:
    attn_slice = cross_attn_last[:, :, :len(tgt_labels), :len(src_labels)]

    fig = plot_attention(attn_slice, src_labels, tgt_labels,
                         title='Cross-Attention (last decoder layer, head 0)')
    plt.savefig('cross_attn_head0.png', dpi=100, bbox_inches='tight')
    plt.show()

    fig = plot_all_heads(attn_slice, src_labels, tgt_labels,
                         title='Cross-Attention – all heads')
    plt.savefig('cross_attn_all_heads.png', dpi=100, bbox_inches='tight')
    plt.show()

print('Attention plots saved.')

FR : je suis dure .
EN : i m kind .
Attention plots saved.


/tmp/ipykernel_21673/636672373.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_21673/636672373.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Encoder self-attention
src_ids_tensor = torch.tensor(src_ids_list, dtype=torch.long).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    _, enc_attn_all = model.encoder(
        src_ids_tensor, src_key_padding_mask=(src_ids_tensor == PAD_ID)
    )
fig = plot_all_heads(enc_attn_all[-1], src_labels, src_labels,
                     title='Encoder Self-Attention (last layer)')
plt.savefig('enc_self_attn.png', dpi=100, bbox_inches='tight')
plt.show()
print('Encoder attention plot saved.')

Encoder attention plot saved.


/tmp/ipykernel_21673/3374907369.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Live Translation

Use the dedicated script in the terminal:

```bash
python live_test.py                             # interactive mode
python live_test.py --sentence "je suis dure ." # single sentence
python live_test.py --sentence "..." --show_attn # with attention heatmap
```